# Silver Layer — Preprocessing & Sample Construction

Validates, cleans, Z-score normalizes, and builds MV AR-LSTM training samples.

**Input:** `bronze_df` from Bronze layer

**Output:** `silver_data` dict with `df`, `data_3d`, `data_norm`, `mean`, `std`, `cities`, `city_onehot`, `dates`, `samples`

## Validate & Clean

In [ ]:
spark_df = spark.read.table("bronze_weather_india.weather.raw_weather")
df = spark_df.toPandas()
df.columns = [c.upper() for c in df.columns]

df["PRECIPITATION_MM"] = df["PRECIPITATION_MM"].clip(lower=0.0)
df = df.drop_duplicates(subset=["CITY", "DATE"], keep="first")
df = df.sort_values(["CITY", "DATE"]).reset_index(drop=True)

print(f"Cleaned: {len(df):,} rows | {df['CITY'].nunique()} regions | "
      f"{df['DATE'].min().date()} -> {df['DATE'].max().date()}")

## Assemble Silver Output

In [ ]:
# -- Spark conversion (commented out for future cluster deployment) ----------
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
silver_sdf = spark.createDataFrame(df)

In [ ]:
# Create silver catalog
silver_catalog = "silver_weather_india"
silver_schema = "weather"
silver_table = "processed_weather"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {silver_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_catalog}.{silver_schema}")

silver_sdf.write \
	.mode("overwrite") \
	.format("csv") \
    .option("header", "true") \
	.option("sep", ",") \
	.option("quote", '"') \
	.option("escape", "\\") \
	.saveAsTable(f"{silver_catalog}.{silver_schema}.{silver_table}")

In [ ]:
df = spark.read.table("silver_weather_india.weather.processed_weather")
df.show()